# CNN por segundo para el FST

**Versión del código: fst_cnn_seg 0.3.0 (red fst_cnn 5.0.0, entrada fusion)**

Corre las celdas **en orden**, una por una (Shift+Enter).

Esta versión deja **figuras listas para la documentación**: datos y desbalance,
muestras de fotogramas por conducta, curva de pérdida en vivo mientras entrena,
matriz de confusión, precisión / recall / F1 por conducta, líneas de tiempo y
ejemplos de errores. Todas se guardan en `Mi unidad/fst/figuras/` (PNG a 200 dpi).

Antes de empezar:
1. Menú **Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU (T4)**.
2. En la PC: `py cnn/recortar_seg.py`. Deja la carpeta `clips_seg`.
3. En tu Google Drive, dentro de `Mi unidad/fst/`, sube:
   - la carpeta `clips_seg`
   - `cnn/fst_cnn.py` **y** `cnn/fst_cnn_seg.py` (el segundo usa el primero)

La **sección 8** trae varias formas de intentar subir la nota sin cambiar lo anterior: suavizar en el tiempo, menos épocas, más regularización, varias redes y más. Las secciones 1 a 7 hacen lo mismo que antes.

## 1. Comprobar que hay GPU

In [ ]:
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO HAY. Cambia el entorno a GPU (ver arriba)')

## 2. Conectar Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

CARPETA = '/content/drive/MyDrive/fst'
CLIPS = CARPETA + '/clips_seg'
CORRIDAS = CARPETA + '/corridas'    # aquí se guardan las predicciones de cada validación
FIGURAS = CARPETA + '/figuras'      # aquí se guardan los PNG y la tabla de métricas
import os, glob
print(len(glob.glob(CLIPS + '/*.npz')), 'videos encontrados en', CLIPS)

## 3. Cargar el código

In [ ]:
import sys, importlib
sys.path.insert(0, CARPETA)
import fst_cnn, fst_cnn_seg
importlib.reload(fst_cnn)
importlib.reload(fst_cnn_seg)

VERSION_ESPERADA = '0.3.0'   # la de este cuaderno
print('Este cuaderno espera fst_cnn_seg', VERSION_ESPERADA)
if fst_cnn_seg.VERSION_SEG != VERSION_ESPERADA:
    print('OJO: Drive tiene la version', fst_cnn_seg.VERSION_SEG, '- sube de nuevo cnn/fst_cnn.py y cnn/fst_cnn_seg.py')
else:
    print('OK: Drive tiene la misma version')

## 4. Los datos, antes de entrenar

Tabla y figura del **desbalance entre conductas**, y fotogramas reales de lo que
ve la red. Esto no entrena nada: tarda un minuto y sirve para la sección de
datos de la documentación.

In [ ]:
X, d, _ = fst_cnn_seg.cargar(CLIPS)
tabla_datos = fst_cnn_seg.figura_datos(d, FIGURAS)
tabla_datos

In [ ]:
# 2 segundos al azar de cada conducta (de videos distintos si se puede), con 4
# fotogramas repartidos en la ventana de 4 s. Cambia semilla para ver otros.
fst_cnn_seg.figura_muestras(X, d, ejemplos=2, fotogramas=4, semilla=0, carpeta_fig=FIGURAS)

## 5. Validar por video (con la curva de pérdida en vivo)

Cada video se predice con una red que **nunca lo vio**. Solo cuenta lo que tú
marcaste en revisor_segundos, así que esta nota sí es honesta.

La curva de pérdida se redibuja después de cada época, una línea por parte de la
validación. Si baja y se aplana, la red aprendió; si una parte queda muy arriba de
las demás, ese video es difícil.

Con ~15 espécimenes tarda alrededor de una hora. Si Colab se corta, cambia a
`cada=2`: entrena con la mitad de los segundos y tarda la mitad.

In [ ]:
fst_cnn_seg.activar_curva_viva()
P, d = fst_cnn_seg.validar(CLIPS, partes=5, cada=1, nombre='base', carpeta_corridas=CORRIDAS)

## 6. Figuras de resultados para la documentación

Precisión, recall y F1 **por conducta** (con la meta de 0.85 marcada), matriz de
confusión, segundos por espécimen del humano contra la red, qué pasa al descartar
los segundos dudosos, líneas de tiempo humano contra red y ejemplos de errores.

In [ ]:
fst_cnn_seg.figura_entrenamiento(carpeta_fig=FIGURAS)      # curva de pérdida final, por parte y media
tabla_metricas = fst_cnn_seg.figuras_resultados(P, d, X, carpeta_fig=FIGURAS)
tabla_metricas

La tabla también queda en `figuras/metricas_por_conducta.csv`. Las notas de arriba
son **segundo contra segundo, solo sobre lo que marcaste tú**. Mira el recuadro de
"Qué pasa al descartar los segundos dudosos" antes de citar una cifra.

## 7. Entrenar la red final con todo y guardarla en Drive

In [ ]:
fst_cnn_seg.activar_curva_viva()
fst_cnn_seg.final(CLIPS, CARPETA + '/fst_cnn_seg.pt', cada=1)
fst_cnn_seg.figura_entrenamiento({'final': fst_cnn_seg.HISTORIA['final']}, FIGURAS)

Después descarga `fst_cnn_seg.pt` y la carpeta `figuras` de Drive y, en la PC:

    py cnn/probar_seg.py etiquetas/NUEVO_rasgos.csv --red fst_cnn_seg.pt

## 8. Probar mejoras

Cada celda de esta sección es **independiente** y se puede correr sola. Ninguna cambia el modelo que ya guardaste en la sección 7: las corridas se guardan en `Mi unidad/fst/corridas/` y la celda 8f las compara todas.

| Celda | Qué prueba | Cuánto tarda (A100) |
|---|---|---|
| 8a | suavizar en el tiempo, modelo de transiciones, contexto de los segundos vecinos | minutos, sin entrenar |
| 8b | menos épocas / más regularización / congelar capas | 25 a 35 min cada una |
| 8c | 3 redes promediadas + volteo al predecir | 1.5 a 2 h |
| 8d | entrenar sin los segundos de frontera entre conductas | 30 min |
| 8e | todo junto, con lo que haya salido mejor | 1.5 a 2 h |
| 8f | tabla comparativa de todas las corridas | segundos |

Compara siempre con la corrida `base` (la de la sección 5): **exactitud 0.716, F1 promedio 0.705**.

### 8a. Sin entrenar: mirar los segundos vecinos

Usa `P` y `d` de la sección 5. Si cerraste Colab, descomenta la primera línea para recuperarlos.

In [ ]:
# P, d = fst_cnn_seg.cargar_corrida(CORRIDAS + '/base.npz')
tabla_pos, variantes = fst_cnn_seg.comparar_posproceso(P, d)
fst_cnn_seg.exactitud_sin_transiciones(P, d)
tabla_pos

Para ver las figuras con la variante que mejor salga, por ejemplo suavizado de ±3 s:

    fst_cnn_seg.figuras_resultados(variantes['suavizado +-3 s'], d, X, carpeta_fig=FIGURAS + '/suavizado')

**Ojo:** el suavizado, el modelo de transiciones y el contexto se aplican *después* de la red. Si alguno gana, el análisis del sistema tendría que aplicarlo también (es un cambio en el programa, no solo un `.pt` nuevo).

### 8b. Menos sobreajuste

La pérdida de entrenamiento baja a 0.22 pero con videos nuevos la exactitud se queda en 0.72: la red memoriza. Aquí se prueba entrenar menos y regularizar más. Primero solo las épocas; luego, todo junto.

In [ ]:
P_ep8, _ = fst_cnn_seg.validar(CLIPS, partes=5, epocas=8, nombre='ep8', carpeta_corridas=CORRIDAS)

In [ ]:
# más regularización (weight_decay 1e-2, label smoothing 0.1) y sin entrenar las 2 primeras etapas de la CNN
P_reg, _ = fst_cnn_seg.validar(CLIPS, partes=5, epocas=8, decay=1e-2, suavizado_etiq=0.1, congelar=2,
                               nombre='reg_ep8', carpeta_corridas=CORRIDAS)

### 8c. Varias redes promediadas y volteo al predecir

Entrena 3 redes por parte con semillas distintas y promedia sus probabilidades, y predice también el clip volteado. Tarda 3 veces más; con 8 épocas son 1.5 a 2 horas.

In [ ]:
P_ens, _ = fst_cnn_seg.validar(CLIPS, partes=5, epocas=8, semillas=3, tta=True,
                               nombre='ens3_tta_ep8', carpeta_corridas=CORRIDAS)

### 8d. Sin los segundos de frontera

El 70 % de los bloques de 5 s tiene más de una conducta, y el segundo donde una cambia a otra es dudoso hasta para ti. Esta prueba no entrena con esos segundos (la prueba sí los usa todos).

In [ ]:
P_sin, _ = fst_cnn_seg.validar(CLIPS, partes=5, epocas=8, quitar_transiciones=True,
                               nombre='sin_frontera_ep8', carpeta_corridas=CORRIDAS)

### 8e. Todo junto

Cuando sepas qué opciones ayudaron, pruébalas combinadas. Ajusta los valores a lo que haya salido mejor en 8b a 8d.

In [ ]:
P_junto, d_junto = fst_cnn_seg.validar(CLIPS, partes=5, epocas=8, decay=1e-2, suavizado_etiq=0.1, congelar=2,
                                       semillas=3, tta=True, quitar_transiciones=True,
                                       nombre='junto', carpeta_corridas=CORRIDAS)
tabla_pos, variantes = fst_cnn_seg.comparar_posproceso(P_junto, d_junto)

### 8f. Comparar todas las corridas

In [ ]:
tabla_corridas = fst_cnn_seg.resumen_corridas(CORRIDAS)
tabla_corridas

### 8g. Entrenar la red final con las opciones ganadoras

Solo si alguna corrida superó a `base`. **Guarda con otro nombre** para no pisar `fst_cnn_seg.pt`. Las opciones son las mismas de `validar`; las que se aplican *después* de la red (suavizado, transiciones, contexto, varias redes, volteo) no se guardan en el `.pt`.

In [ ]:
# fst_cnn_seg.activar_curva_viva()
# fst_cnn_seg.final(CLIPS, CARPETA + '/fst_cnn_seg_prueba.pt', epocas=8, decay=1e-2, suavizado_etiq=0.1, congelar=2)